# Week 5 -- Function 8: PyTorch NN + GP + SVM Active Filter (8D)

In [1]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 40
DATA_PATH_X  = '../data/function_8/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_8/initial_outputs.npy'

weekly_log = [
    ([0.816815, 0.656459, 0.258882, 0.854757, 0.737459, 0.233756, 0.838932, 0.820863], 7.27515),  # W1: 7.275
    ([0.856208, 0.91428, 0.313812, 0.369507, 0.710367, 0.257166, 0.646775, 0.025767], 7.627456),  # W2: 7.627
    ([0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463], 9.038246),  # W3: 9.038
    ([0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0], 9.5183),  # W4: 9.518
    # Week 5: add result here as ([x...], y_value)
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

Synced 1 new observation(s).
X: (44, 8) | Y: (44,)
Week 5 | 44 total observations (40 initial + 4 added)


In [2]:
# Cell 3: Load data and inspect -- Function 8: Portfolio Optimisation (8D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 72)
print('  All observations (sorted descending by Y)')
print('=' * 72)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.6f}' for v in x_val)
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:+.6e}{marker}')
print('=' * 72)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.8f}' for v in best_X)
print(f'\n  Best Y*  = {best_Y:.6e}')
print(f'  Best X*  = [{best_x_str}]')

Input  shape : (44, 8)
Output shape : (44,)

  All observations (sorted descending by Y)
  [ 1]  X = [0.056447, 0.065956, 0.022929, 0.038786, 0.403935, 0.801055, 0.488307, 0.893085]   Y = +9.598482e+00  <-- best
  [ 2]  X = [0.000000, 0.000000, 0.000000, 0.000000, 1.000000, 1.000000, 0.000000, 1.000000]   Y = +9.518300e+00
  [ 3]  X = [0.192640, 0.630677, 0.416796, 0.490529, 0.796086, 0.654567, 0.276241, 0.295518]   Y = +9.344274e+00
  [ 4]  X = [0.481245, 0.102461, 0.219486, 0.677322, 0.247509, 0.244341, 0.163825, 0.715962]   Y = +9.183005e+00
  [ 5]  X = [0.145120, 0.119328, 0.420888, 0.387609, 0.155423, 0.875172, 0.510560, 0.728611]   Y = +9.141639e+00
  [ 6]  X = [0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463]   Y = +9.038246e+00
  [ 7]  X = [0.044329, 0.013581, 0.258198, 0.577644, 0.051280, 0.158563, 0.591030, 0.077953]   Y = +9.013075e+00
  [ 8]  X = [0.143550, 0.937415, 0.232325, 0.009043, 0.414579, 0.409325, 0.553779, 0.205841]   Y = +8.976554e+

In [3]:
# Cell 4: Fit GP
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

Y_fit = np.log(np.abs(Y) + 1e-300)
kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_log = np.log(np.abs(best_Y) + 1e-300)
print(f'  Sanity check at best X* : GP mean={pred_mean[0]:.4f}  actual={actual_log:.4f}  std={pred_std[0]:.6f}')
print('=' * 62)

  GP Fitting Results
  Kernel                  : RBF(length_scale=0.1)
  Log-marginal-likelihood : -130.1899
  Sanity check at best X* : GP mean=2.2616  actual=2.2616  std=0.000010


In [4]:
# Cell 5: PyTorch NN Surrogate (Assignment 16.1 style)
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

y_scaler = StandardScaler()
Y_scaled = y_scaler.fit_transform(Y.reshape(-1, 1)).ravel().astype(np.float32)

X_t = torch.tensor(X.astype(np.float32))
Y_t = torch.tensor(Y_scaled)

class SurrogateNN(nn.Module):
    def __init__(self, n_in, h1=128, h2=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, h1), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(h2, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = SurrogateNN(n_dims)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)

for epoch in range(2000):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(X_t), Y_t)
    loss.backward()
    optimizer.step()
    if epoch % 300 == 0:
        print(f'  Epoch {epoch:4d}  MSE = {loss.item():.4f}')

print(f'\n  Final MSE: {loss.item():.4f}')
model.eval()
with torch.no_grad():
    nn_pred_best = model(torch.tensor(best_X.astype(np.float32))).item()
print(f'  NN at best X*: {nn_pred_best:.4f} (scaled)')

  Epoch    0  MSE = 0.9902


  Epoch  300  MSE = 0.0317


  Epoch  600  MSE = 0.0209


  Epoch  900  MSE = 0.0133


  Epoch 1200  MSE = 0.0126


  Epoch 1500  MSE = 0.0119


  Epoch 1800  MSE = 0.0112



  Final MSE: 0.0151
  NN at best X*: 1.6479 (scaled)


In [5]:
# Cell 6: Dual Acquisition -- NN gradient ascent vs GP UCB
# SVM will pick the final candidate in the next cell.
import torch

trust_radius = 0.1
lo = np.clip(best_X - trust_radius, 0.0, 1.0).astype(np.float32)
hi = np.clip(best_X + trust_radius, 0.0, 1.0).astype(np.float32)
lo_t = torch.tensor(lo)
hi_t = torch.tensor(hi)

# Strategy A: NN gradient ascent
x_opt = torch.tensor(best_X.copy().astype(np.float32), requires_grad=True)
opt_inner = torch.optim.Adam([x_opt], lr=5e-3)
model.eval()
for step in range(800):
    opt_inner.zero_grad()
    (-model(x_opt)).backward()
    opt_inner.step()
    with torch.no_grad():
        x_opt.clamp_(lo_t, hi_t)
grad_candidate = x_opt.detach().numpy().copy()

# Strategy B: GP
np.random.seed(42)
X_grid = np.random.uniform(0, 1, size=(30000, n_dims))
gp_mean, gp_std = gp.predict(X_grid, return_std=True)
ucb_scores = gp_mean + 1.0 * gp_std
gp_candidate = X_grid[np.argmax(ucb_scores)]

# NN scores for both (used as fallback if SVM finds both poor)
model.eval()
with torch.no_grad():
    nn_score_grad = model(torch.tensor(grad_candidate.astype(np.float32))).item()
    nn_score_gp   = model(torch.tensor(gp_candidate.astype(np.float32))).item()

# Preliminary winner by NN score (SVM may override)
selected = 'NN gradient ascent' if nn_score_grad >= nn_score_gp else 'GP UCB'
next_x   = grad_candidate if nn_score_grad >= nn_score_gp else gp_candidate
portal_string = '-'.join([f'{v:.6f}' for v in next_x])

print('=' * 72)
print('  Dual Acquisition (preliminary -- SVM filter next)')
print('=' * 72)
grad_str = ', '.join(f'{v:.6f}' for v in grad_candidate)
gp_str   = ', '.join(f'{v:.6f}' for v in gp_candidate)
print(f'  NN gradient candidate : [{grad_str}]  (NN score: {nn_score_grad:.4f})')
print(f'  GP UCB candidate      : [{gp_str}]  (NN score: {nn_score_gp:.4f})')
print(f'  Preliminary winner    : {selected}')
print('=' * 72)

  Dual Acquisition (preliminary -- SVM filter next)
  NN gradient candidate : [0.000000, 0.034176, 0.000000, 0.000000, 0.303935, 0.701055, 0.388307, 0.946118]  (NN score: 1.7378)
  GP UCB candidate      : [0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463]  (NN score: 1.1773)
  Preliminary winner    : NN gradient ascent


In [6]:
# Cell 7: Input Gradient Sensitivity
import torch

x_sens = torch.tensor(best_X.copy().astype(np.float32), requires_grad=True)
model.eval()
model(x_sens).backward()
grads = x_sens.grad.abs().numpy()

best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'Input gradient magnitudes at best X* = [{best_x_str}]:')
print()
max_g = grads.max() if grads.max() > 0 else 1.0
for i, g in enumerate(grads):
    bar = '#' * max(1, int(g * 40 / max_g))
    print(f'  x{i+1}: |grad| = {g:.4f}  {bar}')
print(f'\n  Most sensitive: x{int(np.argmax(grads))+1}')

Input gradient magnitudes at best X* = [0.056447, 0.065956, 0.022929, 0.038786, 0.403935, 0.801055, 0.488307, 0.893085]:

  x1: |grad| = 0.7114  ########################################
  x2: |grad| = 0.0442  ##
  x3: |grad| = 0.4287  ########################
  x4: |grad| = 0.2543  ##############
  x5: |grad| = 0.0352  #
  x6: |grad| = 0.1254  #######
  x7: |grad| = 0.2287  ############
  x8: |grad| = 0.0044  #

  Most sensitive: x1


In [7]:
# Cell 8: SVM Active Filter
# Scores both candidates and picks the one with higher P(good).
# If both flagged poor (P < 0.3), falls back to NN-score winner.
from sklearn.svm import SVC

threshold = np.percentile(Y, 70)
labels = (Y >= threshold).astype(int)

print(f'  SVM threshold (70th pct): {threshold:.6e}')
print(f'  Good samples: {labels.sum()} / {len(labels)}')
print()

if labels.sum() >= 2 and (labels == 0).sum() >= 2:
    svm_clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    svm_clf.fit(X, labels)

    prob_grad = svm_clf.predict_proba(grad_candidate.reshape(1, -1))[0, 1]
    prob_gp   = svm_clf.predict_proba(gp_candidate.reshape(1, -1))[0, 1]

    print(f'  P(good) -- NN gradient candidate : {prob_grad:.3f}')
    print(f'  P(good) -- GP candidate          : {prob_gp:.3f}')
    print()

    if max(prob_grad, prob_gp) < 0.3:
        svm_selected = selected + ' (SVM fallback -- both poor, keeping NN winner)'
        print(f'  Both candidates flagged poor. Keeping NN winner: {selected}')
    elif prob_grad >= prob_gp:
        next_x = grad_candidate.copy()
        svm_selected = 'NN gradient ascent (SVM endorsed)'
        portal_string = '-'.join([f'{v:.6f}' for v in next_x])
        print(f'  SVM selects: NN gradient ascent (P={prob_grad:.3f})')
    else:
        next_x = gp_candidate.copy()
        svm_selected = 'GP candidate (SVM endorsed)'
        portal_string = '-'.join([f'{v:.6f}' for v in next_x])
        print(f'  SVM selects: GP candidate (P={prob_gp:.3f})')
else:
    svm_selected = selected + ' (insufficient SVM data)'
    print('  Insufficient class balance for SVM -- keeping NN winner.')

next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'\n  FINAL query : [{next_str}]')
print(f'  Portal      : >>> {portal_string} <<<')

  SVM threshold (70th pct): 8.460827e+00
  Good samples: 13 / 44

  P(good) -- NN gradient candidate : 0.946
  P(good) -- GP candidate          : 0.950

  SVM selects: GP candidate (P=0.950)

  FINAL query : [0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463]
  Portal      : >>> 0.076274-0.101214-0.383035-0.338493-0.113685-0.882235-0.615428-0.796463 <<<


In [8]:
print('=' * 72)
print('  SUMMARY -- Week 5 Bayesian Optimisation')
print('=' * 72)
print('  Function        : 8 -- Portfolio Optimisation (8D)')
print('  Objective       : Maximise')
print(f'  NN architecture : MLP [{n_dims} -> 128 -> 128 -> 1], Adam lr=1e-3, 2000 epochs')
print(f'  Acquisition     : NN gradient ascent vs GP, SVM active filter')
print(f'  SVM decision    : {svm_selected}')
print()
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X*         : [{best_x_s}]')
print(f'  Best Y*         : {best_Y:.6e}')
print()
print(f'  Next query      : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

  SUMMARY -- Week 5 Bayesian Optimisation
  Function        : 8 -- Portfolio Optimisation (8D)
  Objective       : Maximise
  NN architecture : MLP [8 -> 128 -> 128 -> 1], Adam lr=1e-3, 2000 epochs
  Acquisition     : NN gradient ascent vs GP, SVM active filter
  SVM decision    : GP candidate (SVM endorsed)

  Best X*         : [0.056447, 0.065956, 0.022929, 0.038786, 0.403935, 0.801055, 0.488307, 0.893085]
  Best Y*         : 9.598482e+00

  Next query      : [0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463]

  Portal submission string:
  >>> 0.076274-0.101214-0.383035-0.338493-0.113685-0.882235-0.615428-0.796463 <<<


## Week 5 -- Reflection

**Function**: F8 -- Portfolio Optimisation (8D)

### Strategy note
W4 gave 9.518. Exploit near W4 with NN+GP dual. SVM active filter.

### Query
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 6
*(fill in after reflecting on result)*